# Experiment: WOP Python vs C++ Comparison

This notebook runs both implementations and compares estimator values and runtime.

Expected workspace layout:
- `external_wop` (Python package)
- `external_wop_cpp` (C++ project)


In [ ]:
from __future__ import annotations

import json
import shutil
import statistics
import subprocess
import sys
import time
from pathlib import Path


def locate_workspace_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "external_wop").exists() and (candidate / "external_wop_cpp").exists():
            return candidate
    raise FileNotFoundError("Could not find workspace root with external_wop and external_wop_cpp")


WORKSPACE_ROOT = locate_workspace_root(Path.cwd())
PY_ROOT = WORKSPACE_ROOT / "external_wop"
CPP_ROOT = WORKSPACE_ROOT / "external_wop_cpp"

print(f"WORKSPACE_ROOT = {WORKSPACE_ROOT}")
print(f"PY_ROOT       = {PY_ROOT}")
print(f"CPP_ROOT      = {CPP_ROOT}")


## Prepare C++ CLI

This cell configures and builds `wop_cli` in `external_wop_cpp/build`.


In [ ]:
cmake_exe = shutil.which("cmake")
if cmake_exe is None:
    cmake_candidates = [
        Path("C:/Program Files/CMake/bin/cmake.exe"),
        Path("C:/Program Files (x86)/CMake/bin/cmake.exe"),
    ]
    cmake_exe = next((str(p) for p in cmake_candidates if p.exists()), None)

if cmake_exe is None:
    raise FileNotFoundError(
        "CMake is not available in PATH. Install CMake and restart Jupyter, "
        "or add cmake.exe to PATH."
    )

print(f"Using CMake: {cmake_exe}")

subprocess.run([
    cmake_exe, "-S", ".", "-B", "build", "-DCMAKE_BUILD_TYPE=Release"
], cwd=CPP_ROOT, check=True)

subprocess.run([
    cmake_exe, "--build", "build", "--config", "Release"
], cwd=CPP_ROOT, check=True)

cpp_candidates = [
    CPP_ROOT / "build" / "Release" / "wop_cli.exe",
    CPP_ROOT / "build" / "Debug" / "wop_cli.exe",
    CPP_ROOT / "build" / "wop_cli.exe",
    CPP_ROOT / "build" / "wop_cli",
]

CPP_CLI = next((p for p in cpp_candidates if p.exists()), None)
if CPP_CLI is None:
    raise FileNotFoundError("wop_cli was not found in external_wop_cpp/build")

CPP_CLI


## Comparison Helpers

Python run uses `python -m wop.cli` from `external_wop`.
C++ run uses `external_wop_cpp` CLI with `--json`.


In [ ]:
INT_KEYS = {"n_total", "n_truncated"}
FLOAT_KEYS = {"J", "exact", "abs_error", "S2", "eps", "mean_steps"}


def parse_python_cli(raw: str) -> dict[str, float | int]:
    parsed: dict[str, float | int] = {}
    for line in raw.splitlines():
        if ":" not in line:
            continue
        key, value = line.split(":", 1)
        key = key.strip()
        value = value.strip()
        if key in INT_KEYS:
            parsed[key] = int(float(value))
        elif key in FLOAT_KEYS:
            parsed[key] = float(value)
    return parsed


def run_python_once(n_paths: int, seed: int, x0: tuple[float, float, float] = (3.0, 0.0, 0.0)) -> dict[str, float | int]:
    cmd = [
        sys.executable,
        "-m",
        "wop.cli",
        "--example",
        "box",
        "--x0",
        f"{x0[0]} {x0[1]} {x0[2]}",
        "--n",
        str(int(n_paths)),
        "--seed",
        str(int(seed)),
    ]
    t0 = time.perf_counter()
    out = subprocess.check_output(cmd, cwd=PY_ROOT, text=True)
    elapsed = time.perf_counter() - t0
    parsed = parse_python_cli(out)
    parsed["runtime_sec"] = elapsed
    return parsed


def run_cpp_once(n_paths: int, seed: int, x0: tuple[float, float, float] = (3.0, 0.0, 0.0)) -> dict[str, float | int]:
    cmd = [
        str(CPP_CLI),
        "--example",
        "box",
        "--x0",
        f"{x0[0]} {x0[1]} {x0[2]}",
        "--n",
        str(int(n_paths)),
        "--seed",
        str(int(seed)),
        "--json",
    ]
    t0 = time.perf_counter()
    out = subprocess.check_output(cmd, cwd=CPP_ROOT, text=True)
    elapsed = time.perf_counter() - t0
    parsed = json.loads(out)
    parsed["runtime_sec"] = elapsed
    return parsed


## Run Comparative Tests

Adjust `N_VALUES` and `SEEDS` if needed.


In [ ]:
N_VALUES = [2000, 5000, 10000]
SEEDS = [314159, 271828, 161803]

rows: list[dict[str, float | int]] = []

for n_paths in N_VALUES:
    for seed in SEEDS:
        py = run_python_once(n_paths=n_paths, seed=seed)
        cpp = run_cpp_once(n_paths=n_paths, seed=seed)
        rows.append({
            "n_paths": n_paths,
            "seed": seed,
            "J_py": float(py["J"]),
            "J_cpp": float(cpp["J"]),
            "abs_delta_J": abs(float(py["J"]) - float(cpp["J"])),
            "eps_py": float(py["eps"]),
            "eps_cpp": float(cpp["eps"]),
            "runtime_py_sec": float(py["runtime_sec"]),
            "runtime_cpp_sec": float(cpp["runtime_sec"]),
            "speedup_cpp_vs_py": float(py["runtime_sec"]) / float(cpp["runtime_sec"]),
        })

rows[:3]


In [ ]:
def summarize_by_n(input_rows: list[dict[str, float | int]]) -> list[dict[str, float | int]]:
    grouped: dict[int, list[dict[str, float | int]]] = {}
    for row in input_rows:
        grouped.setdefault(int(row["n_paths"]), []).append(row)

    summary: list[dict[str, float | int]] = []
    for n_paths in sorted(grouped):
        chunk = grouped[n_paths]
        summary.append({
            "n_paths": n_paths,
            "mean_abs_delta_J": statistics.fmean(float(r["abs_delta_J"]) for r in chunk),
            "mean_runtime_py_sec": statistics.fmean(float(r["runtime_py_sec"]) for r in chunk),
            "mean_runtime_cpp_sec": statistics.fmean(float(r["runtime_cpp_sec"]) for r in chunk),
            "mean_speedup_cpp_vs_py": statistics.fmean(float(r["speedup_cpp_vs_py"]) for r in chunk),
        })
    return summary


summary_rows = summarize_by_n(rows)
summary_rows


In [ ]:
try:
    import pandas as pd
    display(pd.DataFrame(rows))
    display(pd.DataFrame(summary_rows))
except Exception as exc:
    print("pandas is optional; raw rows shown above")
    print(exc)
